<a href="https://colab.research.google.com/github/D2718281828nis/LLM_agent-FireCrawl-Graph/blob/main/FireCrawl-CR-site-parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Агентный сбор и анализ клинических рекомендаций

Notebook выполняет единый воспроизводимый сценарий:

1. Firecrawl Agent обходит динамический реестр Минздрава и сохраняет `ID`, `Наименование`, `Дата размещения КР`, `МКБ-10` в CSV.
2. Пользователь выбирает ID; Firecrawl извлекает алгоритмы действий врача из соответствующей рекомендации.
3. Mistral формирует структурированное резюме **только по извлечённому тексту**.

> **Медицинское предупреждение:** результат предназначен для исследовательского прототипа, может быть неполным и не заменяет официальный документ или решение врача. Всегда сверяйте вывод с исходной рекомендацией.


## 1. Установка зависимостей

Ключи должны называться `FIRECRAWL_API_KEY` и `MISTRAL_API_KEY` в Colab Secrets. Не вставляйте значения ключей в код или вывод Notebook.


In [ ]:
%pip install -q --upgrade firecrawl-py mistralai pandas pydantic


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import pandas as pd
from firecrawl import Firecrawl
from google.colab import userdata
from mistralai import Mistral
from pydantic import BaseModel, Field

REGISTRY_URL = "https://cr.minzdrav.gov.ru/clin-rec"
DETAIL_URL_TEMPLATE = "https://cr.minzdrav.gov.ru/view-cr/{clinical_id}"
CSV_PATH = Path("clinical_recommendations_minzdrav.csv")
REQUIRED_COLUMNS = ["ID", "Наименование", "Дата размещения КР", "МКБ-10"]

def require_secret(name: str) -> str:
    value = userdata.get(name)
    if not value:
        raise RuntimeError(
            f"Секрет {name} не найден. Добавьте его в Colab Secrets "
            "и включите Notebook access."
        )
    return value

firecrawl = Firecrawl(api_key=require_secret("FIRECRAWL_API_KEY"))
mistral = Mistral(api_key=require_secret("MISTRAL_API_KEY"))
print("✅ Клиенты настроены; значения секретов не выведены.")


## 2. Контракты данных и совместимость SDK

Pydantic-схемы заставляют агента вернуть машинно-читаемые поля. Вспомогательные функции нормализуют ответы разных версий Firecrawl SDK (`dict` или объект).


In [ ]:
class RegistryRow(BaseModel):
    id: str = Field(description="Точный ID, включая суффиксы и подчёркивания")
    name: str = Field(description="Наименование клинической рекомендации")
    publication_date: str = Field(description="Дата размещения КР как на сайте")
    mkb10: str = Field(description="Код или список кодов МКБ-10 как на сайте")


class RegistryDataset(BaseModel):
    rows: list[RegistryRow]


class DoctorActions(BaseModel):
    title: str = Field(description="Наименование рекомендации")
    source_url: str
    source_sections: list[str] = Field(
        description="Заголовки разделов, из которых извлечены действия"
    )
    actions_text: str = Field(
        description="Полный текст алгоритмов и действий врача без резюмирования"
    )


def as_dict(result: Any) -> dict[str, Any]:
    """Convert Firecrawl/Pydantic responses to a plain dictionary."""
    if isinstance(result, dict):
        return result
    if hasattr(result, "model_dump"):
        return result.model_dump()
    if hasattr(result, "dict"):
        return result.dict()
    data = getattr(result, "data", None)
    if data is not None:
        return as_dict(data)
    raise TypeError(f"Неожиданный тип ответа: {type(result).__name__}")


def payload(result: Any) -> dict[str, Any]:
    """Unwrap the common Firecrawl {'data': ...} response envelope."""
    value = as_dict(result)
    return as_dict(value["data"]) if "data" in value else value


## 3. Обход реестра и CSV

Реестр является динамическим. Firecrawl Agent получает явную инструкцию дождаться таблицы и пройти пагинацию до конца, не выдумывая отсутствующие значения. Повторный запуск обновляет CSV.


In [ ]:
def scrape_registry(client: Firecrawl = firecrawl) -> pd.DataFrame:
    prompt = f"""
Открой официальный реестр {REGISTRY_URL}. Дождись загрузки таблицы.
Последовательно пройди ВСЕ страницы пагинации (кнопка следующей страницы), пока она
не станет недоступна. Для каждой строки верни точные значения четырёх столбцов:
ID, Наименование, Дата размещения КР, МКБ-10. Не сокращай текст, не переводи его
и не выдумывай пропущенные значения; для пустой ячейки верни пустую строку.
Удали дубликаты строк с одинаковым ID.
""".strip()

    result = client.agent(
        prompt=prompt,
        urls=[REGISTRY_URL],
        schema=RegistryDataset.model_json_schema(),
    )
    rows = payload(result).get("rows", [])
    if not rows:
        raise RuntimeError("Firecrawl Agent не вернул ни одной строки реестра.")

    frame = pd.DataFrame(
        {
            "ID": str(row.get("id", "")).strip(),
            "Наименование": str(row.get("name", "")).strip(),
            "Дата размещения КР": str(row.get("publication_date", "")).strip(),
            "МКБ-10": str(row.get("mkb10", "")).strip(),
        }
        for row in rows
    )
    frame = frame.loc[:, REQUIRED_COLUMNS]
    frame = frame[(frame["ID"] != "") & (frame["Наименование"] != "")]
    frame = frame.drop_duplicates(subset="ID", keep="last").reset_index(drop=True)
    if frame.empty:
        raise RuntimeError("После проверки данных реестр оказался пустым.")
    return frame


registry_df = scrape_registry()
registry_df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
print(f"✅ Сохранено {len(registry_df)} строк: {CSV_PATH.resolve()}")
display(registry_df.head(10))


In [ ]:
# Скачать результат из Colab.
from google.colab import files
files.download(str(CSV_PATH))


## 4. Выбор ID и извлечение действий врача

ID валидируется по уже собранному реестру, поэтому произвольный URL создать нельзя. Агент ищет прежде всего «Приложение Б. Алгоритмы действий врача», а также связанные разделы диагностики, лечения, диспансерного наблюдения и критериев срочного направления.


In [ ]:
def choose_clinical_id(frame: pd.DataFrame) -> str:
    known_ids = set(frame["ID"].astype(str))
    while True:
        clinical_id = input("Введите ID клинической рекомендации: ").strip()
        if clinical_id in known_ids:
            row = frame.loc[frame["ID"].astype(str) == clinical_id].iloc[0]
            print(f"Выбрано: {row['Наименование']}")
            return clinical_id
        print("ID отсутствует в загруженном CSV. Проверьте значение и повторите ввод.")


def extract_doctor_actions(
    clinical_id: str, client: Firecrawl = firecrawl
) -> DoctorActions:
    if not re.fullmatch(r"[A-Za-zА-Яа-яЁё0-9_-]+", clinical_id):
        raise ValueError("ID содержит недопустимые символы.")
    url = DETAIL_URL_TEMPLATE.format(clinical_id=clinical_id)
    prompt = f"""
Проанализируй только официальный документ по адресу {url}. Найди «Приложение Б.
Алгоритмы действий врача» и извлеки его полный текст без пересказа. Если приложение
отсутствует или недоступно, извлеки точные фрагменты с действиями врача из разделов
диагностики, лечения, медицинской помощи, диспансерного наблюдения и критериев
срочного направления. Сохрани порядок, условия, отрицания, дозировки и уровни
рекомендаций. Перечисли реально использованные заголовки разделов. Не добавляй
медицинские сведения, которых нет на странице. source_url должен быть равен {url}.
""".strip()
    result = client.agent(
        prompt=prompt, urls=[url], schema=DoctorActions.model_json_schema()
    )
    actions = DoctorActions.model_validate(payload(result))
    if not actions.actions_text.strip():
        raise RuntimeError("Не удалось извлечь действия врача из документа.")
    return actions


clinical_id = choose_clinical_id(registry_df)
actions = extract_doctor_actions(clinical_id)
print(f"✅ Извлечено символов: {len(actions.actions_text)}")
print(f"Источник: {actions.source_url}")
print("Разделы:", "; ".join(actions.source_sections))


## 5. Agentic summary с Mistral

Mistral получает исходный текст и строгие правила: не дополнять его внешними знаниями, отделять условия и указывать пробелы. Ответ сохраняет ссылку на официальный документ для проверки.


In [ ]:
SUMMARY_SYSTEM_PROMPT = """
Ты — аналитический агент по клиническим рекомендациям. Работай ИСКЛЮЧИТЕЛЬНО с
переданным текстом. Не ставь диагноз, не назначай лечение и не дополняй ответ
внешними медицинскими знаниями. Если данных нет, явно пиши «не указано в
извлечённом фрагменте». Сохраняй отрицания, условия, дозировки и срочность.
Ответ дай на русском языке в Markdown со структурой:
1. Цель алгоритма
2. Последовательность действий врача (нумерованный список)
3. Условия и точки принятия решений (если → то)
4. Красные флаги и срочные действия
5. Контроль, наблюдение и критерии эскалации
6. Что требует проверки в полном официальном документе
В конце добавь предупреждение, что это исследовательское резюме, а не медицинская
рекомендация.
""".strip()


def summarize_actions(actions: DoctorActions, client: Mistral = mistral) -> str:
    source = json.dumps(actions.model_dump(), ensure_ascii=False, indent=2)
    response = client.chat.complete(
        model="mistral-small-latest",
        temperature=0,
        messages=[
            {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
            {"role": "user", "content": f"Сформируй резюме этого извлечения:\n{source}"},
        ],
    )
    summary = response.choices[0].message.content
    if not summary:
        raise RuntimeError("Mistral вернул пустое резюме.")
    return summary


summary = summarize_actions(actions)
print(summary)
print(f"\nОфициальный источник: {actions.source_url}")


## 6. Сохранение аудита

Помимо CSV сохраняются необработанное извлечение и резюме. Это позволяет эксперту сопоставить вывод Mistral с текстом, который получил агент.


In [ ]:
audit_path = Path(f"clinical_recommendation_{clinical_id}_analysis.json")
audit_path.write_text(
    json.dumps(
        {
            "clinical_id": clinical_id,
            "extraction": actions.model_dump(),
            "summary": summary,
            "disclaimer": "Исследовательский прототип; требуется проверка врачом и по официальному документу.",
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print(f"✅ Аудит сохранён: {audit_path.resolve()}")
files.download(str(audit_path))
